# Short Data overview

In [ ]:
import polars as pl

from moonpies.consts import DATA_DIR


In [ ]:
modis = pl.read_csv(DATA_DIR/"modis/modis_2020_Austria.csv")
viirs = pl.read_csv(DATA_DIR/"viirs/viirs-jpss1_2020_Austria.csv")
snpp = pl.read_csv(DATA_DIR/"viirs-snpp/viirs-snpp_2020_Austria.csv")

In [ ]:
modis.head(5)

In [ ]:
modis["type"].unique()

In [ ]:
viirs.filter(viirs["type"] == 2)

In [ ]:
viirs.head()

In [ ]:
modis["version"].unique()

In [ ]:
snpp.head()

In [ ]:
import plotly.express as px

In [ ]:
import plotly.express as px

# Created using Mistral Vibe
# Animated map: one frame per acq_date (acq_time ignored), colour by satellite,
# marker size scaled by frp.
modis_anim = (
    modis.with_columns(pl.col("acq_date").str.to_date("%Y-%m-%d")).sort("acq_date")
)
fig_modis = px.scatter_map(
    data_frame=modis_anim,
    lat="latitude",
    lon="longitude",
    color="satellite",
    size="frp",
    size_max=30,
    animation_frame="acq_date",
    center={"lat": 47.7, "lon": 13.8},
    map_style="carto-positron",
    zoom=6,
    height=600,
    title="MODIS hotspots 2020 (Austria) - animated by acq_date",
    hover_data=["frp"],
)
fig_modis.show()


In [ ]:
# Created using Mistral Vibe
# VIIRS animation: SNPP + JPSS-1 combined, one frame per acq_date, colour by
# satellite, marker size scaled by frp.
viirs_all = (
    pl.concat([viirs, snpp])
    .with_columns(pl.col("acq_date").str.to_date("%Y-%m-%d"))
    .sort("acq_date")
)
fig_viirs = px.scatter_map(
    data_frame=viirs_all,
    lat="latitude",
    lon="longitude",
    color="satellite",
    size="frp",
    size_max=30,
    animation_frame="acq_date",
    center={"lat": 47.7, "lon": 13.8},
    map_style="carto-positron",
    zoom=6,
    height=600,
    title="VIIRS hotspots 2020 (Austria) - animated by acq_date",
    hover_data=["frp"],
)
fig_viirs.show()


In [ ]:
# Created using Mistral Vibe
# Merge MODIS + VIIRS onto a common lat/lon grid (0.01 deg ~ 1.1 km), then
# animate the merged detections by acq_date. Detections falling into the same
# grid cell on the same day (per satellite) are aggregated: frp is summed.
GRID_RES = 0.01


def to_grid(df: pl.DataFrame):
    if df["acq_date"].dtype == pl.String:
        df = df.with_columns(pl.col("acq_date").str.to_date("%Y-%m-%d"))
    return (
        df.with_columns(
            lat_cell=(pl.col("latitude") / GRID_RES).round(0).cast(pl.Int64),
            lon_cell=(pl.col("longitude") / GRID_RES).round(0).cast(pl.Int64),
        )
        .group_by(["acq_date", "satellite", "instrument", "lat_cell", "lon_cell"])
        .agg(
            latitude=(pl.col("lat_cell").first() * GRID_RES),
            longitude=(pl.col("lon_cell").first() * GRID_RES),
            frp=pl.col("frp").sum(),
            n_detections=pl.len(),
        )
        .drop("lat_cell", "lon_cell")
    )


merged = pl.concat([to_grid(modis), to_grid(viirs_all)]).sort("acq_date")

fig_merged = px.scatter_map(
    data_frame=merged,
    lat="latitude",
    lon="longitude",
    color="satellite",
    size="frp",
    size_max=30,
    animation_frame="acq_date",
    center={"lat": 47.7, "lon": 13.8},
    map_style="carto-positron",
    zoom=6,
    height=600,
    title="Merged MODIS + VIIRS hotspots 2020 (Austria), 0.01° grid",
    hover_data=["frp", "n_detections", "instrument"],
)
fig_merged.show()
